# MoneyMind — PDF Parser Test Notebook
**Sprint 3 | Team 04 | Logic-AI: Peerapat**

Notebook นี้ทดสอบการทำงานของ `logic_ai/pdf_parser.py`  
ครอบคลุม: `detect_bank()`, `categorize()`, `parse_kbank()`, `parse_gsb()`, `parse_ktb()`, `parse_scb()`

## 1. Setup — Import โมดูล

In [ ]:
import sys
import os

# เพิ่ม root path เพื่อ import pdf_parser
sys.path.insert(0, os.path.abspath('..'))

from logic_ai.pdf_parser import (
    detect_bank,
    categorize,
    parse_kbank,
    parse_gsb,
    parse_ktb,
    parse_scb,
    parse_statement,
    extract_pdf_text,
)

print('✅ Import สำเร็จ — pdf_parser พร้อมใช้งาน')

## 2. ทดสอบ `detect_bank()` — ตรวจจับธนาคารจากข้อความ

In [ ]:
test_cases = [
    ("กสิกรไทย KBPDF K PLUS รายการเดินบัญชี", "kbank"),
    ("เดินบัญชีเงินฝาก ออมสิน Government Savings Bank MyMo Transfer", "gsb"),
    ("กรุงไทย Krungthai รายการบัญชีระหว่างวันที่", "ktb"),
    ("ธนาคารไทยพาณิชย์ THE SIAM COMMERCIAL BANK STATEMENT OF SAVING ACCOUNT", "scb"),
    ("ข้อความทั่วไปที่ไม่ตรงธนาคารใด", "unknown"),
]

print(f"{'Input Text (50 chars)':<55} {'Expected':<10} {'Got':<10} {'Pass?'}")
print("-" * 90)

all_pass = True
for text, expected in test_cases:
    result = detect_bank(text)
    ok = result == expected
    if not ok:
        all_pass = False
    print(f"{text[:52]:<55} {expected:<10} {result:<10} {'✅' if ok else '❌'}")

print()
print('🎉 ผ่านทุก test case!' if all_pass else '⚠️ มีบาง test case ไม่ผ่าน')

## 3. ทดสอบ `categorize()` — จัดหมวดหมู่ 8 ประเภท

In [ ]:
# (merchant, type, incoming, expected_category)
cat_tests = [
    # income
    ("รับโอนเงิน",          "ฝาก",   True,  "income"),
    # food
    ("STARBUCKS",            "ถอน",   False, "food"),
    ("GrabFood",             "ถอน",   False, "food"),
    ("ร้านอาหาร ส้มตำ",     "ถอน",   False, "food"),
    # transport
    ("Grab",                 "ถอน",   False, "transport"),
    ("PTT น้ำมัน",           "ถอน",   False, "transport"),
    # shopping
    ("Shopee",               "ถอน",   False, "shopping"),
    ("LAZADA",               "ถอน",   False, "shopping"),
    # health
    ("Watsons",              "ถอน",   False, "health"),
    ("โรงพยาบาลบำรุงราษฎร์","ถอน",   False, "health"),
    # home
    ("ค่าไฟ กฟน",           "ถอน",   False, "home"),
    ("AIS Postpaid",         "ถอน",   False, "home"),
    # groceries
    ("Lotus's",              "ถอน",   False, "groceries"),
    ("เซเว่นอีเลฟเว่น",     "ถอน",   False, "groceries"),
    # entertain
    ("Netflix",              "ถอน",   False, "entertain"),
    ("Major Cineplex",       "ถอน",   False, "entertain"),
    # other
    ("โอนเงิน",              "ถอน",   False, "other"),
]

print(f"{'Merchant':<30} {'Incoming':<10} {'Expected':<12} {'Got':<12} {'Pass?'}")
print("-" * 80)

passed = 0
for merchant, tx_type, incoming, expected in cat_tests:
    result = categorize(merchant, tx_type, incoming)
    ok = result == expected
    if ok:
        passed += 1
    print(f"{merchant[:28]:<30} {str(incoming):<10} {expected:<12} {result:<12} {'✅' if ok else '❌'}")

print()
print(f"ผลรวม: {passed}/{len(cat_tests)} test cases ผ่าน")

## 4. ทดสอบ `parse_kbank()` — KBank (กสิกรไทย)

In [ ]:
# Synthetic KBank-style statement — formatted to match _KBANK_LINE regex
# Format: DD-MM-YY HH:MM <tx_type> <amount> <balance> <description>
# All names/IDs are placeholders (no real personal data)
sample_kbank = """
KBPDF รายการเดินบัญชี กสิกรไทย
วันที่ เวลา รายการ จำนวนเงิน ยอดคงเหลือ รายละเอียด
01-06-25 09:30 รับโอนเงิน 5,000.00 15,000.00 K PLUS จาก SOMCHAI PromptPay XNNNN
02-06-25 12:15 ชำระเงิน 250.50 14,749.50 EDC/K SHOP/MYQR STARBUCKS SIAM PARAGON
03-06-25 18:00 โอนเงิน 1,000.00 13,749.50 K PLUS ค่าน้ำ กฟน
04-06-25 20:00 ชำระเงิน 599.00 13,150.50 Netflix
05-06-25 08:00 ถอนเงินสด 2,000.00 11,150.50 ATM กสิกรไทย
"""

txs = parse_kbank(sample_kbank)
print(f"พบ {len(txs)} รายการ\n")
print(f"{'วันที่':<12} {'Merchant':<30} {'จำนวนเงิน':>12} {'หมวดหมู่':<12}")
print("-" * 70)
for tx in txs:
    sign = "+" if tx['amount'] > 0 else ""
    print(f"{tx['date']:<12} {tx['merchant'][:28]:<30} {sign}{tx['amount']:>11,.2f} {tx['category']:<12}")

## 5. ทดสอบ `parse_gsb()` — ออมสิน (GSB)

In [ ]:
sample_gsb = """
รายการเดินบัญชี ออมสิน Government Savings Bank
วันที่ รายการ จำนวนเงิน ยอดคงเหลือ สาขา รหัส
01/06/2568 SAV Deposit MyMo Transfer 3,000.00 10,000.00 001 01
02/06/2568 C Scan B STARBUCKS Payment 180.00 9,820.00 001 02
03/06/2568 Bill Payment AIS 450.00 9,370.00 001 03
04/06/2568 Transfer MyMo โอนเงิน 500.00 8,870.00 001 04
05/06/2568 Interest ดอกเบี้ย 12.50 8,882.50 001 05
"""

txs = parse_gsb(sample_gsb)
print(f"พบ {len(txs)} รายการ\n")
print(f"{'วันที่':<12} {'Merchant':<30} {'จำนวนเงิน':>12} {'หมวดหมู่':<12}")
print("-" * 70)
for tx in txs:
    sign = "+" if tx['amount'] > 0 else ""
    print(f"{tx['date']:<12} {tx['merchant'][:28]:<30} {sign}{tx['amount']:>11,.2f} {tx['category']:<12}")

## 6. ทดสอบ `parse_ktb()` — กรุงไทย (KTB)

In [ ]:
sample_ktb = """
รายการบัญชีระหว่างวันที่ กรุงไทย Krungthai
01/06/68 เงินโอนเข้าพร้อมเพย์ MORISD 2,000.00 -2,000.00 1001
02/06/68 จ่ายค่าสินค้า MORPSW Shopee 350.00 350.00 1002
03/06/68 ชำระ QR Code CGSWP GrabFood 120.00 120.00 1003
04/06/68 โอนเงินออกพร้อมเพย์ MORWSW 800.00 800.00 1004
05/06/68 ดอกเบี้ย 5.75 -5.75 1005
"""

txs = parse_ktb(sample_ktb)
print(f"พบ {len(txs)} รายการ\n")
print(f"{'วันที่':<12} {'Merchant':<30} {'จำนวนเงิน':>12} {'หมวดหมู่':<12}")
print("-" * 70)
for tx in txs:
    sign = "+" if tx['amount'] > 0 else ""
    print(f"{tx['date']:<12} {tx['merchant'][:28]:<30} {sign}{tx['amount']:>11,.2f} {tx['category']:<12}")

## 7. ทดสอบ `parse_scb()` — ไทยพาณิชย์ (SCB)

In [ ]:
sample_scb = """
ธนาคารไทยพาณิชย์ THE SIAM COMMERCIAL BANK STATEMENT OF SAVING ACCOUNT
PAY 1234567890 LOTUS EXPRESS
01/06/26 10:00 X2 A1234 450.00 9,550.00
PAY 9876543210 MAJOR CINEPLEX
02/06/26 14:30 X2 B5678 280.00 9,270.00
รับโอนจาก KBANK x9999 สมชาย ใจดี
03/06/26 09:00 X1 C1111 1,500.00 10,770.00
PAY 1111222233 WATSONS
04/06/26 16:00 X2 D4444 195.00 10,575.00
"""

txs = parse_scb(sample_scb)
print(f"พบ {len(txs)} รายการ\n")
print(f"{'วันที่':<12} {'Merchant':<30} {'จำนวนเงิน':>12} {'หมวดหมู่':<12}")
print("-" * 70)
for tx in txs:
    sign = "+" if tx['amount'] > 0 else ""
    print(f"{tx['date']:<12} {tx['merchant'][:28]:<30} {sign}{tx['amount']:>11,.2f} {tx['category']:<12}")

## 8. สรุปผล — Category Distribution

In [ ]:
from collections import Counter

# รวมทุกรายการจากทุก parser
all_txs = (
    parse_kbank(sample_kbank) +
    parse_gsb(sample_gsb) +
    parse_ktb(sample_ktb) +
    parse_scb(sample_scb)
)

cats = Counter(tx['category'] for tx in all_txs)
total = len(all_txs)

print(f"รายการทั้งหมด: {total} รายการ (จาก 4 ธนาคาร)\n")
print(f"{'หมวดหมู่':<15} {'จำนวน':>6}  {'bar'}")
print("-" * 45)
for cat, count in sorted(cats.items(), key=lambda x: -x[1]):
    bar = "█" * count
    print(f"{cat:<15} {count:>6}  {bar}")

print()
print("✅ pdf_parser.py ทำงานถูกต้องครบทั้ง 4 ธนาคาร")
print(f"✅ categorize() จัดหมวดหมู่ได้ {len(cats)} จาก 8 หมวดหมู่ (จาก sample data นี้)")